# `long` / `wide` — Advanced Reference

`long` melts all numeric columns to rows. `wide` pivots a column's values into headers.

| Method | Key params | Purpose |
|---|---|---|
| `.long(col='variable', value='value')` | `col`, `value` | wide → long (melt) |
| `.wide(col='variable', value='value', aggfunc=None)` | `col`, `value`, `aggfunc` | long → wide (pivot) |

They are designed to work together as a pair, but each is also useful standalone.

---

In [9]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
healthexp = pt.sample_data['healthexp']
flights = pt.sample_data['flights']

In [10]:
penguins

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,None
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,None
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


## `long` — melt all numeric columns to rows

In [11]:
# Melt all numeric columns to rows — default col/value names
penguins.long().sample(10)


,species,island,sex,variable,value
605,Gentoo,Biscoe,Male,bill_depth_mm,16.0
156,Chinstrap,Dream,Male,bill_length_mm,52.7
588,Gentoo,Biscoe,Female,bill_depth_mm,13.1
1268,Gentoo,Biscoe,Female,body_mass_g,4150.0
364,Adelie,Biscoe,Female,bill_depth_mm,18.3
812,Adelie,Torgersen,Female,flipper_length_mm,186.0
1085,Adelie,Biscoe,Male,body_mass_g,4050.0
206,Chinstrap,Dream,Female,bill_length_mm,42.5
897,Chinstrap,Dream,Male,flipper_length_mm,203.0
792,Adelie,Biscoe,Female,flipper_length_mm,193.0


In [12]:
# Custom column names for the variable and value columns
penguins.long(col='metric', value='reading').sample(10)


,species,island,sex,metric,reading
1178,Adelie,Dream,Male,body_mass_g,4250.0
21,Adelie,Biscoe,Male,bill_length_mm,37.7
453,Adelie,Biscoe,Male,bill_depth_mm,19.0
973,Gentoo,Biscoe,Male,flipper_length_mm,230.0
679,Gentoo,Biscoe,Male,bill_depth_mm,16.0
335,Gentoo,Biscoe,Male,bill_length_mm,55.1
1207,Chinstrap,Dream,Male,body_mass_g,3800.0
861,Chinstrap,Dream,Male,flipper_length_mm,191.0
593,Gentoo,Biscoe,Male,bill_depth_mm,15.3
166,Chinstrap,Dream,Female,bill_length_mm,45.9


In [13]:
# Only non numeric columns survive as id columns — numerics all get melted
# select narrows which numerics get melted
(penguins
 .select(['species', 'island', 'bill_length_mm', 'bill_depth_mm'])
 .long(col='metric', value='reading')
 .sample(12)
)


,species,island,metric,reading
508,Chinstrap,Dream,bill_depth_mm,17.3
152,Chinstrap,Dream,bill_length_mm,46.5
497,Chinstrap,Dream,bill_depth_mm,19.5
662,Gentoo,Biscoe,bill_depth_mm,14.4
14,Adelie,Torgersen,bill_length_mm,34.6
273,Gentoo,Biscoe,bill_length_mm,50.1
289,Gentoo,Biscoe,bill_length_mm,50.7
256,Gentoo,Biscoe,bill_length_mm,42.6
482,Adelie,Dream,bill_depth_mm,16.5
425,Adelie,Torgersen,bill_depth_mm,17.6


## `wide` — pivot a column's values into headers

In [14]:
# flights has one row per year+month — no aggfunc needed
flights.wide(col='month', value='passengers').head()


,year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,1949,112,118,132,129,121,135,148,148,136,119,104,118
1,1950,115,126,141,135,125,149,170,170,158,133,114,140
2,1951,145,150,178,163,172,178,199,199,184,162,146,166
3,1952,171,180,193,181,183,218,230,242,209,191,172,194
4,1953,196,196,236,235,229,243,264,272,237,211,180,201


In [15]:
# Select only the columns needed so island becomes the clean index
(penguins
 .select(['island', 'species', 'body_mass_g'])
 .wide(col='species', value='body_mass_g', aggfunc='mean')
)


,island,Adelie,Chinstrap,Gentoo
0,Biscoe,3709.659091,NaN,5076.01626
1,Dream,3688.392857,3733.088235,NaN
2,Torgersen,3706.372549,NaN,NaN


## Round-trip: `long` → transform → `wide`
A common pattern: go long to apply uniform operations across all metrics, then come back wide.

In [16]:
# Select only needed columns → long → wide: yields mean bill metrics per island
(penguins
 .select(['island', 'bill_length_mm', 'bill_depth_mm'])
 .long(col='metric', value='reading')
 .wide(col='metric', value='reading', aggfunc='mean')
)


,island,bill_depth_mm,bill_length_mm
0,Biscoe,15.874850,45.257485
1,Dream,18.344355,44.167742
2,Torgersen,18.429412,38.950980
